# Warriner Arousal Robustness Check

This notebook reruns the frame-aware Intensity analysis with Warriner et al. arousal norms. It uses the Warriner collocate handoff produced by the companion Sentiment robustness notebook, so preprocessing is fixed across valence and arousal.


## Setup

Warriner arousal is reported on the native 1-9 scale and as a 0-1 scaled value. The analysis keeps the same annual units, frame strata, separate baselines, and compact trend-model convention as the main Intensity notebook.


In [8]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
INTERIM_WARRINER_DIR = PROJECT_ROOT / "data/interim/lsc/warriner_vad"
WARRINER_MATCHES_PATH = INTERIM_WARRINER_DIR / "lsc_warriner_collocate_matches.parquet"
WARRINER_CONTEXT_COVERAGE_PATH = INTERIM_WARRINER_DIR / "lsc_warriner_context_coverage.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/lsc/intensity/robustness_warriner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANNUAL_AROUSAL_PATH = OUTPUT_DIR / "lsc_intensity_warriner_annual_arousal.csv"
COVERAGE_PATH = OUTPUT_DIR / "lsc_intensity_warriner_coverage.csv"
TOP_COLLOCATES_PATH = OUTPUT_DIR / "lsc_intensity_warriner_top_collocates.csv"
AUDIT_FLAGS_PATH = OUTPUT_DIR / "lsc_intensity_warriner_audit_flags.csv"
TREND_SUMMARY_PATH = OUTPUT_DIR / "lsc_intensity_warriner_trend_models.csv"
TREND_COMPARISON_PATH = OUTPUT_DIR / "lsc_intensity_warriner_nrc_trend_comparison.csv"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_YEARS = list(range(2014, 2027))
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BOOTSTRAP_REPETITIONS = 500
RANDOM_SEED = 123
LOW_MATCHED_TOKEN_COVERAGE_WARN = 0.25
LOW_CONTEXT_COVERAGE_WARN = 0.45
TOP_COLLOCATE_SHARE_WARN = 0.20
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75


## Load Warriner Handoff

The handoff must be created by `02_warriner_valence_robustness_check.ipynb`. Reusing it ensures that valence and arousal differ only by the Warriner score dimension.


In [9]:
if not WARRINER_MATCHES_PATH.exists():
    raise FileNotFoundError(
        f"Missing Warriner collocate handoff: {WARRINER_MATCHES_PATH}. "
        "Run notebooks/02_sentiment/02_warriner_valence_robustness_check.ipynb first."
    )
if not WARRINER_CONTEXT_COVERAGE_PATH.exists():
    raise FileNotFoundError(
        f"Missing Warriner coverage handoff: {WARRINER_CONTEXT_COVERAGE_PATH}. "
        "Run notebooks/02_sentiment/02_warriner_valence_robustness_check.ipynb first."
    )

warriner_matches = pd.read_parquet(WARRINER_MATCHES_PATH)
context_coverage = pd.read_parquet(WARRINER_CONTEXT_COVERAGE_PATH)
required_match_columns = {
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "term_role",
    "target_group",
    "doc_id",
    "collocate",
    "arousal_warriner",
    "arousal_scaled_0_1",
}
required_coverage_columns = {
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "context_row_id",
    "doc_id",
    "candidate_collocate_tokens",
    "matched_warriner_units",
    "matched_token_positions",
    "has_warriner_match",
}
missing_match_columns = sorted(required_match_columns - set(warriner_matches.columns))
missing_coverage_columns = sorted(required_coverage_columns - set(context_coverage.columns))
if missing_match_columns:
    raise RuntimeError(f"Warriner match handoff is missing columns: {missing_match_columns}")
if missing_coverage_columns:
    raise RuntimeError(f"Warriner coverage handoff is missing columns: {missing_coverage_columns}")

observed_units = sorted(context_coverage["analysis_unit"].dropna().unique())
observed_years = sorted(context_coverage["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units in Warriner handoff: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years in Warriner handoff: {missing_years}")

print(f"Warriner match rows: {len(warriner_matches):,}")
print(f"Warriner coverage rows: {len(context_coverage):,}")
warriner_matches.head()


Warriner match rows: 1,202,749
Warriner coverage rows: 311,030


,context_row_id,source_context_row_id,context_id,doc_id,lsc_year,analysis_unit,term_role,target_group,raw_form,frame_stratum,predicted_derived_frame,collocate,collocate_type,surface_text,token_start_in_window,token_end_in_window,matched_token_count,valence_warriner,arousal_warriner,dominance_warriner,valence_scaled_0_1,arousal_scaled_0_1,dominance_scaled_0_1,source_terms,source_term_count
0,0,11401,NaN,0004cb4b97b156b2,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,feeling,unigram,feelings,0,1,1,6.50,3.86,6.50,0.68750,0.35750,0.68750,feeling,1
1,0,11401,NaN,0004cb4b97b156b2,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,continue,unigram,continued,2,3,1,5.61,2.90,6.17,0.57625,0.23750,0.64625,continue,1
2,0,11401,NaN,0004cb4b97b156b2,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,failure,unigram,failure,3,4,1,2.15,5.00,5.04,0.14375,0.50000,0.50500,failure,1
3,1,11402,NaN,000750cc054d9ff4,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,husband,unigram,husband,2,3,1,7.41,4.38,5.72,0.80125,0.42250,0.59000,husband,1
4,1,11402,NaN,000750cc054d9ff4,2014,frustration,baseline,baseline,frustration,unframed_baseline,NaN,moment,unigram,moment,8,9,1,6.10,4.05,6.00,0.63750,0.38125,0.62500,moment,1


## Annual Arousal Index

Annual intensity scores are frequency-weighted means over matched local collocates.


In [10]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_warriner_units_coverage=("matched_warriner_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_warriner_match=("has_warriner_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_warriner_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_arousal = (
    warriner_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        arousal_warriner_mean=("arousal_warriner", "mean"),
        arousal_warriner_sd=("arousal_warriner", "std"),
        arousal_scaled_mean=("arousal_scaled_0_1", "mean"),
        arousal_scaled_sd=("arousal_scaled_0_1", "std"),
        valence_warriner_mean_for_reference=("valence_warriner", "mean"),
        valence_scaled_mean_for_reference=("valence_scaled_0_1", "mean"),
        dominance_warriner_mean_for_reference=("dominance_warriner", "mean"),
        matched_warriner_units=("arousal_warriner", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_arousal = annual_arousal.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_arousal = annual_arousal.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_warriner_mean,arousal_warriner_sd,arousal_scaled_mean,arousal_scaled_sd,valence_warriner_mean_for_reference,valence_scaled_mean_for_reference,dominance_warriner_mean_for_reference,matched_warriner_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_warriner_units_coverage,matched_token_positions,contexts_with_warriner_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,4.133537,0.813490,0.391692,0.101686,5.398560,0.549820,5.355730,4869,966,800,1286,808,9069,4869,4871,1259,0.537104,0.979005,False
1,2015,ADHD,clinical_only,target,ADHD,4.113055,0.794703,0.389132,0.099338,5.391706,0.548963,5.392948,4398,957,759,1203,774,8360,4398,4400,1175,0.526316,0.976725,False
2,2016,ADHD,clinical_only,target,ADHD,4.114192,0.788149,0.389274,0.098519,5.331729,0.541466,5.358084,4283,946,784,1149,790,8030,4283,4284,1136,0.533499,0.988686,False
3,2017,ADHD,clinical_only,target,ADHD,4.111938,0.811620,0.388992,0.101453,5.334541,0.541818,5.333639,4072,941,744,1099,751,7593,4072,4077,1075,0.536942,0.978162,False
4,2018,ADHD,clinical_only,target,ADHD,4.114714,0.801866,0.389339,0.100233,5.363711,0.545464,5.377294,4527,979,781,1182,787,8408,4527,4532,1168,0.539010,0.988156,False
5,2019,ADHD,clinical_only,target,ADHD,4.125591,0.791406,0.390699,0.098926,5.272373,0.534047,5.336583,3851,852,665,1000,674,7120,3851,3851,983,0.540871,0.983000,False
6,2020,ADHD,clinical_only,target,ADHD,4.089699,0.786576,0.386212,0.098322,5.328416,0.541052,5.348486,3569,800,661,959,668,6722,3569,3572,939,0.531389,0.979145,False
7,2021,ADHD,clinical_only,target,ADHD,4.138756,0.813133,0.392344,0.101642,5.291181,0.536398,5.288857,3456,880,634,928,636,6430,3456,3459,913,0.537947,0.983836,False
8,2022,ADHD,clinical_only,target,ADHD,4.084060,0.820874,0.385507,0.102609,5.344647,0.543081,5.395979,3579,851,627,930,634,6550,3579,3580,916,0.546565,0.984946,False
9,2023,ADHD,clinical_only,target,ADHD,4.122442,0.798786,0.390305,0.099848,5.336126,0.542016,5.341492,2717,755,467,750,477,5142,2717,2718,722,0.528588,0.962667,False


## Bootstrap Confidence Intervals

Document-level bootstrap intervals mirror the main arousal notebook.


In [11]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows: list[dict[str, object]] = []

doc_scores = (
    warriner_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(
        arousal_warriner_sum=("arousal_warriner", "sum"),
        arousal_scaled_sum=("arousal_scaled_0_1", "sum"),
        matched_warriner_units=("arousal_warriner", "size"),
    )
)

for group_values, doc_frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    native_sums = doc_frame["arousal_warriner_sum"].to_numpy(dtype=float)
    scaled_sums = doc_frame["arousal_scaled_sum"].to_numpy(dtype=float)
    unit_counts = doc_frame["matched_warriner_units"].to_numpy(dtype=float)
    n_docs = len(doc_frame)
    native_estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    scaled_estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        native_estimates[repetition] = native_sums[sample_index].sum() / denominator if denominator else np.nan
        scaled_estimates[repetition] = scaled_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_rows.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "arousal_warriner_bootstrap_mean": float(np.nanmean(native_estimates)),
            "arousal_warriner_ci_low": float(np.nanpercentile(native_estimates, 2.5)),
            "arousal_warriner_ci_high": float(np.nanpercentile(native_estimates, 97.5)),
            "arousal_scaled_bootstrap_mean": float(np.nanmean(scaled_estimates)),
            "arousal_scaled_ci_low": float(np.nanpercentile(scaled_estimates, 2.5)),
            "arousal_scaled_ci_high": float(np.nanpercentile(scaled_estimates, 97.5)),
        }
    )

bootstrap = pd.DataFrame(bootstrap_rows)
annual_arousal = annual_arousal.merge(bootstrap, on=GROUP_COLUMNS, how="left")
annual_arousal.to_csv(ANNUAL_AROUSAL_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_warriner_mean,arousal_warriner_sd,arousal_scaled_mean,arousal_scaled_sd,valence_warriner_mean_for_reference,valence_scaled_mean_for_reference,dominance_warriner_mean_for_reference,matched_warriner_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_warriner_units_coverage,matched_token_positions,contexts_with_warriner_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,arousal_warriner_bootstrap_mean,arousal_warriner_ci_low,arousal_warriner_ci_high,arousal_scaled_bootstrap_mean,arousal_scaled_ci_low,arousal_scaled_ci_high
0,2014,ADHD,clinical_only,target,ADHD,4.133537,0.813490,0.391692,0.101686,5.398560,0.549820,5.355730,4869,966,800,1286,808,9069,4869,4871,1259,0.537104,0.979005,False,500,doc_id,4.134335,4.110207,4.160050,0.391792,0.388776,0.395006
1,2015,ADHD,clinical_only,target,ADHD,4.113055,0.794703,0.389132,0.099338,5.391706,0.548963,5.392948,4398,957,759,1203,774,8360,4398,4400,1175,0.526316,0.976725,False,500,doc_id,4.114225,4.086257,4.141006,0.389278,0.385782,0.392626
2,2016,ADHD,clinical_only,target,ADHD,4.114192,0.788149,0.389274,0.098519,5.331729,0.541466,5.358084,4283,946,784,1149,790,8030,4283,4284,1136,0.533499,0.988686,False,500,doc_id,4.114209,4.086134,4.141662,0.389276,0.385767,0.392708
3,2017,ADHD,clinical_only,target,ADHD,4.111938,0.811620,0.388992,0.101453,5.334541,0.541818,5.333639,4072,941,744,1099,751,7593,4072,4077,1075,0.536942,0.978162,False,500,doc_id,4.112201,4.084838,4.141459,0.389025,0.385605,0.392682
4,2018,ADHD,clinical_only,target,ADHD,4.114714,0.801866,0.389339,0.100233,5.363711,0.545464,5.377294,4527,979,781,1182,787,8408,4527,4532,1168,0.539010,0.988156,False,500,doc_id,4.114317,4.087470,4.140907,0.389290,0.385934,0.392613
5,2019,ADHD,clinical_only,target,ADHD,4.125591,0.791406,0.390699,0.098926,5.272373,0.534047,5.336583,3851,852,665,1000,674,7120,3851,3851,983,0.540871,0.983000,False,500,doc_id,4.126944,4.096266,4.154965,0.390868,0.387033,0.394371
6,2020,ADHD,clinical_only,target,ADHD,4.089699,0.786576,0.386212,0.098322,5.328416,0.541052,5.348486,3569,800,661,959,668,6722,3569,3572,939,0.531389,0.979145,False,500,doc_id,4.089402,4.057849,4.118849,0.386175,0.382231,0.389856
7,2021,ADHD,clinical_only,target,ADHD,4.138756,0.813133,0.392344,0.101642,5.291181,0.536398,5.288857,3456,880,634,928,636,6430,3456,3459,913,0.537947,0.983836,False,500,doc_id,4.140065,4.109803,4.171343,0.392508,0.388725,0.396418
8,2022,ADHD,clinical_only,target,ADHD,4.084060,0.820874,0.385507,0.102609,5.344647,0.543081,5.395979,3579,851,627,930,634,6550,3579,3580,916,0.546565,0.984946,False,500,doc_id,4.083707,4.052123,4.116080,0.385463,0.381515,0.389510
9,2023,ADHD,clinical_only,target,ADHD,4.122442,0.798786,0.390305,0.099848,5.336126,0.542016,5.341492,2717,755,467,750,477,5142,2717,2718,722,0.528588,0.962667,False,500,doc_id,4.121745,4.089375,4.157796,0.390218,0.386172,0.394725


## Trend Models

Native and scaled Warriner arousal trajectories receive compact OLS trend summaries. The scaled trend is compared with the main NRC-VAD arousal trend for direction checks.


In [12]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
index_columns = {
    "warriner_arousal_native_1_9": "arousal_warriner_mean",
    "warriner_arousal_scaled_0_1": "arousal_scaled_mean",
}
for group_values, frame in annual_arousal.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    for index_name, value_column in index_columns.items():
        trend_rows.append(
            {
                "analysis_unit": analysis_unit,
                "frame_stratum": frame_stratum,
                "term_role": term_role,
                "target_group": target_group,
                "index_name": index_name,
                "value_column": value_column,
                **fit_trend(frame, value_column),
            }
        )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)

main_trend_path = PROJECT_ROOT / "data/processed/lsc/intensity/lsc_intensity_trend_models.csv"
if main_trend_path.exists():
    main_trends = pd.read_csv(main_trend_path)
    main_subset = main_trends.loc[main_trends["index_name"].eq("arousal_mean")].copy()
    robust_subset = trend_summary.loc[trend_summary["index_name"].eq("warriner_arousal_scaled_0_1")].copy()
    comparison = robust_subset.merge(
        main_subset[
            [
                "analysis_unit",
                "frame_stratum",
                "linear_slope_per_year",
                "linear_p_value",
                "linear_adj_r_squared",
                "autocorrelation_flag",
            ]
        ].rename(
            columns={
                "linear_slope_per_year": "nrc_linear_slope_per_year",
                "linear_p_value": "nrc_linear_p_value",
                "linear_adj_r_squared": "nrc_linear_adj_r_squared",
                "autocorrelation_flag": "nrc_autocorrelation_flag",
            }
        ),
        on=["analysis_unit", "frame_stratum"],
        how="left",
    )
    comparison = comparison.rename(
        columns={
            "linear_slope_per_year": "warriner_scaled_linear_slope_per_year",
            "linear_p_value": "warriner_scaled_linear_p_value",
            "linear_adj_r_squared": "warriner_scaled_linear_adj_r_squared",
            "autocorrelation_flag": "warriner_scaled_autocorrelation_flag",
        }
    )
    comparison["slope_direction_agrees"] = np.sign(comparison["warriner_scaled_linear_slope_per_year"]) == np.sign(
        comparison["nrc_linear_slope_per_year"]
    )
    comparison.to_csv(TREND_COMPARISON_PATH, index=False)
else:
    comparison = pd.DataFrame()

trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,value_column,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,warriner_arousal_native_1_9,arousal_warriner_mean,13,2020.0,4.107476,-0.003501,0.001488,0.038256,0.334887,0.274423,-0.578695,2.746855,-0.629620,False,NaN,NaN,0.290066,0.015644
1,ADHD,clinical_only,target,ADHD,warriner_arousal_scaled_0_1,arousal_scaled_mean,13,2020.0,0.388435,-0.000438,0.000186,0.038256,0.334887,0.274423,-0.578695,2.746855,-0.629620,False,NaN,NaN,0.290066,0.015644
2,ADHD,lived_only,target,ADHD,warriner_arousal_native_1_9,arousal_warriner_mean,13,2020.0,4.049277,-0.000170,0.001904,0.930481,0.000724,-0.090120,-0.026903,1.699917,-0.029564,False,NaN,NaN,0.091654,0.181773
3,ADHD,lived_only,target,ADHD,warriner_arousal_scaled_0_1,arousal_scaled_mean,13,2020.0,0.381160,-0.000021,0.000238,0.930481,0.000724,-0.090120,-0.026903,1.699917,-0.029564,False,NaN,NaN,0.091654,0.181773
4,ADHD,mixed,target,ADHD,warriner_arousal_native_1_9,arousal_warriner_mean,13,2020.0,4.066971,-0.007127,0.002049,0.005170,0.523687,0.480386,-0.723663,2.780451,-0.554130,True,-0.007983,0.000091,0.439357,-0.041029
5,ADHD,mixed,target,ADHD,warriner_arousal_scaled_0_1,arousal_scaled_mean,13,2020.0,0.383371,-0.000891,0.000256,0.005170,0.523687,0.480386,-0.723663,2.780451,-0.554130,True,-0.000998,0.000091,0.439357,-0.041029
6,ADHD,substantive_core_overall,target,ADHD,warriner_arousal_native_1_9,arousal_warriner_mean,13,2020.0,4.090626,-0.003867,0.001003,0.002681,0.574518,0.535838,-0.757970,2.539376,-0.468752,False,NaN,NaN,0.584597,0.048760
7,ADHD,substantive_core_overall,target,ADHD,warriner_arousal_scaled_0_1,arousal_scaled_mean,13,2020.0,0.386328,-0.000483,0.000125,0.002681,0.574518,0.535838,-0.757970,2.539376,-0.468752,False,NaN,NaN,0.584597,0.048760
8,Autism,clinical_only,target,Autism,warriner_arousal_native_1_9,arousal_warriner_mean,13,2020.0,4.130038,-0.000724,0.001697,0.677750,0.016291,-0.073137,-0.127636,2.200501,-0.228211,False,NaN,NaN,-0.174708,-0.101571
9,Autism,clinical_only,target,Autism,warriner_arousal_scaled_0_1,arousal_scaled_mean,13,2020.0,0.391255,-0.000091,0.000212,0.677750,0.016291,-0.073137,-0.127636,2.200501,-0.228211,False,NaN,NaN,-0.174708,-0.101571


## Diagnostics And Saved Outputs

Coverage, top-collocate diagnostics, and audit flags are saved as CSV outputs.


In [13]:
collocate_counts = (
    warriner_matches.groupby([*GROUP_COLUMNS, "collocate"], as_index=False)
    .agg(collocate_count=("collocate", "size"))
)
collocate_counts["group_match_count"] = collocate_counts.groupby(GROUP_COLUMNS)["collocate_count"].transform("sum")
collocate_counts["collocate_share"] = collocate_counts["collocate_count"] / collocate_counts["group_match_count"]
top_share = collocate_counts.sort_values("collocate_share", ascending=False).groupby(GROUP_COLUMNS, as_index=False).head(1)
top_share = top_share.rename(columns={"collocate": "top_collocate", "collocate_share": "top_collocate_share"})[
    [*GROUP_COLUMNS, "top_collocate", "top_collocate_share"]
]

top_collocates = (
    warriner_matches.groupby(["analysis_unit", "frame_stratum", "collocate"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        documents=("doc_id", "nunique"),
        arousal_warriner_mean=("arousal_warriner", "mean"),
        arousal_scaled_mean=("arousal_scaled_0_1", "mean"),
        valence_warriner_mean=("valence_warriner", "mean"),
        source_terms=("source_terms", "first"),
    )
)
top_collocates["unit_frame_occurrences"] = top_collocates.groupby(["analysis_unit", "frame_stratum"])["occurrences"].transform("sum")
top_collocates["collocate_share"] = top_collocates["occurrences"] / top_collocates["unit_frame_occurrences"]
top_collocates = (
    top_collocates.sort_values(["analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, False])
    .groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .head(25)
    .reset_index(drop=True)
)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)

coverage_for_flags = coverage.merge(top_share, on=GROUP_COLUMNS, how="left")
flag_rows = []
for row in coverage_for_flags.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if pd.notna(row.matched_token_coverage) and row.matched_token_coverage < LOW_MATCHED_TOKEN_COVERAGE_WARN:
        flags.append("low_matched_token_coverage")
    if pd.notna(row.context_match_coverage) and row.context_match_coverage < LOW_CONTEXT_COVERAGE_WARN:
        flags.append("low_context_match_coverage")
    if pd.notna(row.top_collocate_share) and row.top_collocate_share > TOP_COLLOCATE_SHARE_WARN:
        flags.append("dominant_top_collocate")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "context_rows": row.context_rows,
                "documents": row.documents,
                "matched_token_coverage": row.matched_token_coverage,
                "context_match_coverage": row.context_match_coverage,
                "top_collocate": row.top_collocate,
                "top_collocate_share": row.top_collocate_share,
            }
        )
audit_flags = pd.DataFrame(flag_rows)
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

pd.DataFrame(
    {
        "output": [
            ANNUAL_AROUSAL_PATH.relative_to(PROJECT_ROOT),
            COVERAGE_PATH.relative_to(PROJECT_ROOT),
            TREND_SUMMARY_PATH.relative_to(PROJECT_ROOT),
            TOP_COLLOCATES_PATH.relative_to(PROJECT_ROOT),
            AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT),
        ]
    }
)


,output
0,data/processed/lsc/intensity/robustness_warriner/lsc_intensity_warriner_annual_arousal.csv
1,data/processed/lsc/intensity/robustness_warriner/lsc_intensity_warriner_coverage.csv
2,data/processed/lsc/intensity/robustness_warriner/lsc_intensity_warriner_trend_models.csv
3,data/processed/lsc/intensity/robustness_warriner/lsc_intensity_warriner_top_collocates.csv
4,data/processed/lsc/intensity/robustness_warriner/lsc_intensity_warriner_audit_flags.csv


## Handoff Summary

This check fails if the annual analysis grid is incomplete.


In [14]:
expected_annual_rows = len(EXPECTED_YEARS) * (len(BASELINE_UNITS) + len(TARGET_UNITS) * len(TARGET_FRAME_STRATA))
summary = {
    "annual_rows": len(annual_arousal),
    "expected_annual_rows": expected_annual_rows,
    "warriner_match_rows": len(warriner_matches),
    "trend_rows": len(trend_summary),
    "audit_flag_rows": len(audit_flags),
    "comparison_rows": len(comparison),
}
if summary["annual_rows"] != expected_annual_rows:
    raise RuntimeError(f"Expected {expected_annual_rows} annual rows, found {summary['annual_rows']}.")
summary


{'annual_rows': 143,
 'expected_annual_rows': 143,
 'warriner_match_rows': 1202749,
 'trend_rows': 22,
 'audit_flag_rows': 1,
 'comparison_rows': 11}